# Experiments Notebook

This notebook is used for running experiments and testing different configurations of the neural network.

In [31]:
import sys
import torch
from pathlib import Path

# Get the current working directory and check where we are
print(f"Current working directory: {Path.cwd()}")

# If you're in the notebooks directory, go up one level
if Path.cwd().name == 'notebooks':
    sys.path.append(str(Path.cwd().parent))
else:
    # If you're in the project root, add current directory
    sys.path.append(str(Path.cwd()))

# Import necessary libraries

from src.layers.input import InputLayer;
from src.layers.dense import DenseLayer;

from src.losses.cross_entropy import Loss;
from src.optimizers.gradient_descent import GD;
from src.utils.data_loader import load_data;




Current working directory: /home/protim/Documents/basis/notebooks


In [32]:
# Define hyperparameters
hyperparams = {
    'input_units': 2,
    'hidden_units': 2,
    'output_units': 1

}

In [33]:
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

input_layer = InputLayer( hyperparams, name='input', device=device );
hidden_layer = DenseLayer( hyperparams, name='hidden', transfer='sigmoid', device=device );
output_layer = DenseLayer( hyperparams, name='output', transfer='sigmoid', device=device );

network = input_layer >> hidden_layer >> output_layer;

In [34]:
# Load XOR data
( train_data, train_labels ), ( test_data, test_labels ) = load_data( "xor", device=device );
print( f"Train data shape: {train_data.shape}, Train labels shape: {train_labels.shape}" );

Train data shape: torch.Size([4, 2]), Train labels shape: torch.Size([4, 1])


In [35]:
print( f"\nNumber of trainable parameters: {sum( p.numel() for p in network.trainable_parameters.values() )}" );
print( f"\nTrainable Parameters:" );
for name, param in network.trainable_parameters.items():
    print( f"{name}: {param.shape}" );

# print param values
print("\nInitial parameter values:");
for name, param in network.trainable_parameters.items():
    print(f"{name}: {param.data}")



Number of trainable parameters: 9

Trainable Parameters:
hidden: torch.Size([2, 1])
W_input_hidden_layer: torch.Size([2, 2])
W_hidden_output_layer: torch.Size([1, 2])
output: torch.Size([1, 1])

Initial parameter values:
hidden: tensor([[0.],
        [0.]], device='cuda:0')
W_input_hidden_layer: tensor([[ 0.1372,  1.5283],
        [-0.1217,  0.6004]], device='cuda:0')
W_hidden_output_layer: tensor([[ 0.1136, -0.0884]], device='cuda:0')
output: tensor([[0.]], device='cuda:0')


In [36]:
print("XOR Network Forward Pass:")
for i in range(train_data.shape[0]):
    input_tensor = train_data[i].unsqueeze(1) # Shape (2, 1)
    output = network.forward(input_tensor)
    print(f"Input: {train_data[i].tolist()} -> Output: {output['output'].squeeze().item():.4f}, Expected: {train_labels[i].item():.1f}")

XOR Network Forward Pass:
Input: [0.0, 0.0] -> Output: 0.5032, Expected: 0.0
Input: [0.0, 1.0] -> Output: 0.5091, Expected: 1.0
Input: [1.0, 0.0] -> Output: 0.5048, Expected: 1.0
Input: [1.0, 1.0] -> Output: 0.5102, Expected: 0.0


In [37]:
# 1. Instantiate Loss Function
bce_loss = Loss(name="binary_cross_entropy")

In [38]:
# 2. Compile the network (Optimizer and Learning Rate)
# Gradients are now averaged over the batch, so the effective step size is ~1/N
# of the old (summed-gradient) behaviour. Use a larger learning rate for XOR
# (4 samples) so it still escapes the sigmoid plateau within a few thousand epochs.
learning_rate = 7.0
network.compile(loss=bce_loss, optimizer_class=GD, learning_rate=learning_rate)

In [39]:
# 3. Train the Network
epochs = 300 # More epochs for XOR
import threading
# Start GPU monitoring in separate thread
print("\nStarting Training...")
network.train(train_data, train_labels, epochs=epochs, save_param_history=True)
print("Training Finished.")


Starting Training...
GPU Available: NVIDIA GeForce RTX 3080
Number of GPUs: 1
TensorBoard log directory created at: runs/nn_training_gpu_cuda/run_20260708_085402_6879366a
Epoch 1/300, Loss: 0.693021, Time: 40.4ms


USDT:2026-07-08 08:54:03 276558:276558 ActivityProfilerController.cpp:415] profiler_start
USDT:2026-07-08 08:54:04 276558:276558 ActivityProfilerController.cpp:455] profiler_stop
USDT:2026-07-08 08:54:05 276558:276558 ActivityProfilerController.cpp:415] profiler_start
USDT:2026-07-08 08:54:06 276558:276558 ActivityProfilerController.cpp:455] profiler_stop


Epoch 100/300, Loss: 0.054452, Time: 22.3ms
Epoch 200/300, Loss: 0.011122, Time: 8.2ms
Epoch 300/300, Loss: 0.006102, Time: 12.9ms

Peak GPU memory usage: 39.6 MB
Total training time: 15.59 seconds
Average time per epoch: 26.42 ms

Profiler trace saved to: runs/nn_training_gpu_cuda/profiler
Training Finished.


In [40]:
# Load TensorBoard extension
%load_ext tensorboard

# Display TensorBoard in the notebook
%tensorboard --logdir ../notebooks/runs/

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 276951), started 4:28:25 ago. (Use '!kill 276951' to kill it.)